# Train and Evaluate Models - BERT, RoBERTa, and BERTweet

In [1]:
# GPU present?
import os, subprocess, sys
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
!nvidia-smi | head -n 15


CUDA_VISIBLE_DEVICES = 0,1
Mon Nov 10 19:48:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:01:00.0 Off |                  Off |
| 30%   27C    P8             28W /  300W |      35MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------

In [2]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Looking in links: https://download.pytorch.org/whl/cu118/torch_stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 116.5 MB/s  0:00:0500:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 43.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.0 MB/s  0:00:00
  Using cached nvidia_cuda_nvrtc_cu11-11.8.89-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu11-11.8.89-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu11-11.8.87-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu11-8.7.0.84-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu11-11.11.3.6-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu11-10.9.0.58-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cura

In [3]:
%pip install --upgrade --no-cache-dir "bitsandbytes==0.43.3"
# if 0.43.3 still complains on your setup, try:
# %pip install "bitsandbytes==0.42.0"


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 80.0 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.48.1
    Uninstalling bitsandbytes-0.48.1:
      Successfully uninstalled bitsandbytes-0.48.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/apps/software/standard/core/jupyterlab/4.4.6-py3.12/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/apps/software/standard/core/jupyterlab/4.4.6-py3.12/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/apps/software/standard/core/jupyterlab/4.4.6-py3.12/lib/python3.12/site-package

Torch version: 2.2.2+cu118
CUDA available: True
GPU: NVIDIA RTX A6000


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Set random seed
SEED=42
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA RTX A6000


In [2]:
#Create tweet dataset class
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [3]:
#Create data loading functions
def load_data(dataset_name, data_dir='./processed_data'):
    train = pd.read_csv(f'{data_dir}/{dataset_name}_train.csv')
    val = pd.read_csv(f'{data_dir}/{dataset_name}_val.csv')
    test = pd.read_csv(f'{data_dir}/{dataset_name}_test.csv')
    
    print(f"Loaded {dataset_name}:")
    print(f"  Train: {len(train):,}")
    print(f"  Val:   {len(val):,}")
    print(f"  Test:  {len(test):,}")
    
    return train, val, test

def create_datasets(train_df, val_df, test_df, tokenizer, 
                       text_col='text_normalized', label_col='sentiment',
                       batch_size=16, max_length=128):
    train_dataset = TweetDataset(
        train_df[text_col].values,
        train_df[label_col].values,
        tokenizer,
        max_length
    )
    
    val_dataset = TweetDataset(
        val_df[text_col].values,
        val_df[label_col].values,
        tokenizer,
        max_length
    )
    
    test_dataset = TweetDataset(
        test_df[text_col].values,
        test_df[label_col].values,
        tokenizer,
        max_length
    )

    # #similar to notes (lecture 4)
    # train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    # val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    # test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_dataset, val_dataset, test_dataset

In [4]:
#Calculate accuracy and F1 scores for eval
def calc_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1
    }

In [5]:
#plotting functions

#shows how model learned over time (loss and accuracy/f1)
def plot_training_history(log_history, model_name):
    train_loss = [x['loss'] for x in log_history if 'loss' in x]
    val_loss = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
    val_accuracy = [x['eval_accuracy'] for x in log_history if 'eval_accuracy' in x]
    val_f1 = [x['eval_f1_macro'] for x in log_history if 'eval_f1_macro' in x]
    
    if not train_loss:
        print("No training history to plot")
        return
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    epochs_train = range(1, len(train_loss) + 1)
    epochs_val = range(1, len(val_loss) + 1)
    ax1.plot(epochs_train, train_loss, marker='o', label='Train Loss')
    ax1.plot(epochs_val, val_loss, marker='s', label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{model_name} - Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Metrics
    ax2.plot(epochs_val, val_accuracy, marker='o', label='Accuracy')
    ax2.plot(epochs_val, val_f1, marker='s', label='F1 (macro)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score')
    ax2.set_title(f'{model_name} - Metrics')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace('-', '_')
    os.makedirs("./models", exist_ok=True)
    plt.savefig(f'./models/{safe_title}_confusion_matrix.png')
    plt.show()

In [12]:
#use HuggingFace API to train model
#Note: Api handles training loop, validation and early stopping, learning rate schedule, and logging
def train_model(model_name, dataset_name, num_labels, 
                      text_col='text_normalized', label_col='sentiment',
                      batch_size=16, learning_rate=2e-5, epochs=5):
    
    print(f"Training: {model_name} on {dataset_name}")
    
    #load data
    train_df, val_df, test_df = load_data(dataset_name)
    
    #initialize tokenizer/model
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False if model_name=="vinai/bertweet-base" else True,
        normalization=True if model_name=="vinai/bertweet-base" else False
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )
    
    print(f"Model Name: {model_name}")
    print(f"Number of labels: {num_labels}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    #datasets
    train_dataset, val_dataset, test_dataset = create_datasets(
        train_df, val_df, test_df, tokenizer, text_col, label_col
    )
    
    #save the model and dataset in output directory
    save_name = f"{model_name.split('/')[-1]}_{dataset_name}"
    output_dir = f"./models/{save_name}"

    #train model
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        report_to="none",
        seed=SEED,
        logging_dir=f'{output_dir}/logs',
        logging_steps=50,               # Log every 50 steps
        disable_tqdm=False,             # Enable progress bars
        log_level='info'                # Show training info
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=calc_metrics
    )
    
    trainer.train()
    print("\nTraining complete!")
    
    #Plot history
    plot_training_history(trainer.state.log_history, save_name)
    
    #save model/tokenizer
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Model and tokenizer saved")
    
    #Eval
    print("Evaluate test results")
    test_results = trainer.predict(test_dataset)
    test_predictions = np.argmax(test_results.predictions, axis=-1)
    test_labels = test_results.label_ids
    
    test_accuracy = accuracy_score(test_labels, test_predictions)
    test_f1 = f1_score(test_labels, test_predictions, average='macro')
    
    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test F1 (macro): {test_f1:.4f}")
    
    if dataset_name == 'sentiment140':
        label_names = ['Negative', 'Positive']
    else:
        with open('./processed_data/ipc_label_mapping.json', 'r') as f:
            label_mapping = json.load(f)
            label_names = [k for k, v in sorted(label_mapping.items(), key=lambda x: x[1])]
    
    #report prints metrics
    print("\nClassification Report:")
    print(classification_report(
        test_labels,
        test_predictions,
        target_names=label_names,
        digits=4
    ))
    
    #Conf matrix
    plot_confusion_matrix(
        test_labels,
        test_predictions,
        label_names,
        f'{save_name} - Confusion Matrix'
    )
    
    #Save results
    results = {
        'model': model_name,
        'dataset': dataset_name,
        'accuracy': float(test_accuracy),
        'f1_macro': float(test_f1)
    }
    
    with open(f'{output_dir}/test_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\nResults saved to {output_dir}/test_results.json")
    
    return model, results

In [ ]:
print("Model Training for BERT model")

all_results = []

#BERT on Sentiment140
print("\n")
print("BERT-base on Sentiment140")
model, results = train_model(
    model_name='bert-base-uncased',
    dataset_name='sentiment140',
    num_labels=2,
    label_col='sentiment'
)
all_results.append(results)



Model Training for BERT model


BERT-base on Sentiment140
Training: bert-base-uncased on sentiment140
Loaded sentiment140:
  Train: 316,583
  Val:   39,573
  Test:  39,573


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Model Name: bert-base-uncased
Number of labels: 2
Total parameters: 109,483,778


***** Running training *****
  Num examples = 316,583
  Num Epochs = 5
  Instantaneous batch size per device = 16
  Training with DataParallel so batch size has been adjusted to: 64
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 1
  Total optimization steps = 24,735
  Number of trainable parameters = 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.353300,0.324773,0.859374,0.859302



***** Running Evaluation *****
  Num examples = 39573
  Batch size = 64
Saving model checkpoint to ./models/bert-base-uncased_sentiment140/checkpoint-4947
Configuration saved in ./models/bert-base-uncased_sentiment140/checkpoint-4947/config.json
Model weights saved in ./models/bert-base-uncased_sentiment140/checkpoint-4947/model.safetensors


In [ ]:
#BERT on Israel-Palestine Conflict: GazaWar kaggle dataset
print("\n")
print("BERT-base on IPC")
model, results = train_model(
    model_name='bert-base-uncased',
    dataset_name='ipc',
    num_labels=6,
    label_col='label_id'
)
all_results.append(results)

In [ ]:
print("Model Training for RoBERTa model")

#RoBERTa on Sentiment140
print("\n")
print("RoBERTa-base on Sentiment140")
model, results = train_model(
    model_name='roberta-base',
    dataset_name='sentiment140',
    num_labels=2,
    label_col='sentiment'
)
all_results.append(results)



Model Training for RoBERTa model


RoBERTa-base on Sentiment140
Training: roberta-base on sentiment140
Loaded sentiment140:
  Train: 316,583
  Val:   39,573
  Test:  39,573


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

  2025-11-07T18:07:19.180618Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7fd057d7bdd0>), traceback: Some(<traceback object at 0x7fceed07c900>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

  2025-11-07T18:07:19.181157Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7fd057d7bdd0>), traceback: Some(<traceback object at 0x7fceed07c840>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28

  2025-11-07T18:07:19.181292Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7fd057d7bdd0>), traceback: Some(<traceback object at 0x7fceed07c440>) }, caller: "src/progress_update.rs:313"
    

In [ ]:
#RoBERTa on Israel-Palestine Conflict: GazaWar kaggle dataset
print("\n")
print("RoBERTa-base on IPC")    
model, results = train_model(
    model_name='roberta-base',
    dataset_name='ipc',
    num_labels=6,
    label_col='label_id'
)
all_results.append(results)

In [ ]:
print("Model Training for BERTweet model")

#BERTweet on Sentiment140
print("\n")
print("BERTweet on Sentiment140")
model, results = train_model(
    model_name='vinai/bertweet-base',
    dataset_name='sentiment140',
    num_labels=2,
    label_col='sentiment'
)
all_results.append(results)

In [ ]:
#BERTweet on Israel-Palestine Conflict: GazaWar kaggle dataset
print("\n")
print("BERTweet on IPC")
model, results = train_model(
    model_name='vinai/bertweet-base',
    dataset_name='ipc',
    num_labels=6,
    label_col='label_id'
)
all_results.append(results)

In [ ]:
#Compare final results
print("Final results:")

comparison_df = pd.DataFrame(all_results)
print("\n", comparison_df.to_string(index=False))

comparison_df.to_csv('./models/final_comparison.csv', index=False)

print("\nAll models complete")
print("Results saved to ./models/")

#when reloading models (sample)
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# model = AutoModelForSequenceClassification.from_pretrained('./models/bert-base-uncased_sentiment140')
# tokenizer = AutoTokenizer.from_pretrained('./models/bert-base-uncased_sentiment140')